Title image

"Custom model training in WorldCereal"
PART 2: Training and applying your model

### Content

- [Introduction](###-Introduction)
- [How to run this notebook?](###-How-to-run-this-notebook?)
- [Before you start](###-Before-you-start)

ADD OTHER SECTIONS

### Introduction

Overall goal of the demo

Short recap of part 1 (extractions of private ref data)

Explanation of part 2 (actual model training and inference)

### How to run this notebook?

#### Option 1: Run on Terrascope

You can use a preconfigured environment on [**Terrascope**](https://terrascope.be/en) to run the workflows in a Jupyter notebook environment. Just register as a new user on Terrascope or use one of the supported EGI eduGAIN login methods to get started.

Once you have a Terrascope account, you can run this notebook by clicking the button shown below.

<div class="alert alert-block alert-warning">When you click the button, you will be prompted with "Server Options". Make sure to select the "Worldcereal" image here. Did you choose "Terrascope" by accident? Then go to File > Hub Control Panel > Stop my server, and click the link below once again.</div>


ADD DEMO BUTTON


#### Option 2: Install Locally

If you prefer to install the package locally, you can create the WorldCereal environment using **Conda** or **pip**.

First clone the repository:
```bash
git clone https://github.com/WorldCereal/worldcereal-classification.git
cd worldcereal-classification
```
Next, install the package locally:
- for Conda: `conda env create -f environment.yml`
- for Pip: `pip install .[train,notebooks]`

### 1. Gather your training data

Training data can be sourced from:
- publicly available extractions --> query_public_extractions
- private extractions (PART 1 of this exercise) --> query_private_extractions

Both public and private extractions can be queried using:

- collection id
- spatial extent
- temporal extent
- list of crop types


#### 1.1. Querying individual datasets

Easiest example is to just use all training data associated with one or multiple datasets.

In [ ]:
ONLY SHOWCASE THIS FOR PRIVATE DATASETS, NOT FOR PUBLIC

private_data = query_private_extractions(ref_ids = [])

training_data = merge_data(public_data, private_data)

#### 1.2. Querying training data based on spatial extent, temporal extent and/or crop types



**Step 1: select your area of interest**

When running the code snippet below, an interactive map will be visualized.
Click the Rectangle button on the left hand side of the map to start drawing your region of interest.
The widget will automatically store the coordinates of the last rectangle you drew on the map.

In [ ]:
from worldcereal.utils.map import ui_map

map = ui_map()
map.show_map()


specify buffer for querying extractions

**Step 2: select your season of interest**

In [ ]:
from notebook_utils.seasons import retrieve_worldcereal_seasons

spatial_extent = map.get_extent()
seasons = retrieve_worldcereal_seasons(spatial_extent)

Now use the slider to select your processing period. Note that the length of the period is always fixed to a year.
Just make sure your season of interest is fully captured within the period you select.

In [1]:
from notebook_utils.dateslider import date_slider

slider = date_slider()

**Step 3: select your crops of interest**

Will we support this?

**Step 4: Run the query**


Now we query our local extractions database and retrieve the relevant samples based on defined area of interest and processing period.

You can also include training data from public datasets by using the "include_public" flag.

This will result in one giant geopandas geodataframe containing all the relevant extractions.

In [ ]:
from notebook_utils.extractions import query_extractions

# Retrieve the polygon you just drew
polygon = map.get_polygon_latlon()

# Retrieve the date range you just selected
processing_period = slider.get_processing_period()

# Define whether you want to include public data or not
# TODO: Setting this to "True" won't work at the moment!
include_public = False
include_private = True


query_public_extractions --> via duckdb --> use phaseI extractions !!!
query_private_extractions --> via duckdb


extract_df = query_extractions(polygon,
                               processing_period=processing_period,
                               include_public=include_public,
                               include_private=include_private,
                               croptypes=croptypes,
                               private_extraction_dir=extractions_dir,)
extract_df.head()

#### 1.3. Check the training data



In [ ]:
# You can inspect the timeseries for a specific point or points:

from notebook_utils.extractions import visualize_timeseries

# By default this generates a plot of NDVI and first point in the dataframe
outfile = outfolder_col / 'timeseries_plot.png'
visualize_timeseries(extract_df, outfile)


### 2. Prepare your training data

- ensuring all required data attributes are there

(check progress christina after monday!)
- run process_parquet to get the data in the correct format --> ensuring focus time is set correctly?

In [ ]:
# most of the following code is just there temporarily, in order to be able to use the process_parquet function as it was implemented for phase I extractions
# TODO: process_parquet should be adapted to work with the new format of the extractions

# Convert extraction dataframe into the format that is required for model training
import numpy as np
import pandas as pd

# create label columns
extract_df['CROPTYPE_LABEL'] = extract_df['ewoc_code'].values
extract_df['LANDCOVER_LABEL'] = np.zeros(extract_df.shape[0])
extract_df['POTAPOV-LABEL-10m'] = np.zeros(extract_df.shape[0])
extract_df['WORLDCOVER-LABEL-10m'] = np.zeros(extract_df.shape[0])

# create DEM columns
extract_df.rename(columns={'elevation': 'DEM-alt-20m'}, inplace=True)
extract_df.rename(columns={'slope': 'DEM-slo-20m'}, inplace=True)

# rename valid_time to valid_date
extract_df['valid_time'] = pd.to_datetime(extract_df['valid_time'])
extract_df.rename(columns={'valid_time': 'valid_date'}, inplace=True)

# others
extract_df['location_id'] = extract_df['sample_id'].values
extract_df['aez_zoneid'] = np.zeros(extract_df.shape[0])

# here we actually run process_parquet:
from worldcereal.utils.refdata import process_parquet
# Prepare training dataframe
training_df = process_parquet(extract_df)
training_df.head()

Now the user needs to select which crop types to include and which ones to merge...

In [ ]:
from utils import CropTypePicker

croptypepicker = CropTypePicker(training_df, samples_threshold=2)
croptypepicker.display()


In [ ]:
# now apply the crop type picker and generate the new dataframe, only containing the classes of interest
# (contained in attribute "final_class")

# QUESTION: in the previous implementation, all non-selected classes were assigned to other_temporary_crops.
# Should we keep this or just remove the non-selected classes from the dataframe?
# currently the latter has been implemented

new_df = croptypepicker.apply()
new_df['final_class'].value_counts()

Once the final dataframe has been compiled, we compute the Presto features (process_parquet)

In [ ]:
# Prepare training features

from utils import prepare_training_dataframe

training_dataframe = prepare_training_dataframe(new_df, task_type="croptype")
training_dataframe.head()

### 3. Custom model training and deployment

In [ ]:
# Train model

from utils import train_classifier

custom_model, report, confusion_matrix = train_classifier(training_dataframe, balance_classes=False)
# Print the classification report
print(report)

In [ ]:
# deploy model

from worldcereal.utils.upload import deploy_model
from openeo_gfmap.backend import cdse_connection
from utils import get_input

modelname = get_input("model")
model_url = deploy_model(cdse_connection(), custom_model, pattern=modelname)
print(f"Your model can be downloaded from: {model_url}")

### 4. Run inference and show resulting map

In [ ]:
# Run inference and visualize output...